## **Chemical Equilibrium: select your reaction**

<div align="left">
  <table border="1" cellpadding="6" cellspacing="0">
    <tr>
      <td bgcolor="#444444">
        <font color="#ffeb3b"><tt><b>Last updated (YYYY-MM-DD): 2026-04-17</b></tt></font>
      </td>
    </tr>
  </table>
</div>



In this notebook, you will analyze **a gas-phase elementary reaction of your choice** as a concrete case study to explore the fundamental principles governing chemical equilibrium. You will begin by specifying the reaction stoichiometry (reactants, products, and stoichiometric coefficients). The notebook then uses that input to build a unified computational workflow for predicting equilibrium behavior.

The equilibrium will be examined from two complementary perspectives: **classical thermodynamics**, which provides macroscopic equilibrium criteria through the minimization of appropriate thermodynamic potentials; and **statistical thermodynamics**, which connects equilibrium properties to molecular-level information (e.g., energies and partition functions).
<!--
and chemical kinetics, which describes the time-dependent evolution of the reacting system and its dynamic approach to equilibrium.
-->

The notebook is designed to guide you through a quantitative description of equilibrium by (i) analyzing how thermodynamic potentials vary with the extent of reaction, (ii) determining equilibrium compositions from minimization criteria under specified constraints, and (iii) exploring how changes in temperature, pressure, or initial composition shift the equilibrium position. Overall, the goal is to present equilibrium not as a static condition, but as a **predictable outcome** that can be understood consistently from thermodynamic,
and molecular viewpoints within a single computational framework.


### **Preparing the Computational Environment**  


*The next two cells prepare the environment by installing the necessary packages, importing libraries, and loading constants and functions. Make sure to run hem sequentially before doing anything else.*

⏳ **Note:** This may take **a few minutes** to complete. ⏳  

In [ ]:
#@title <small> 💻 Install Libraries (this may take a while) <small> { display-mode: "form" }
%%capture captured_output

# --- system (for LaTeX text in matplotlib) --- #
! sudo apt update -y
! sudo apt install -y cm-super dvipng texlive-latex-extra texlive-latex-recommended

# --- Python: install and update core scientific libraries --- #

# versions that are known to work well together will be selected to avoid compatibility issues
%pip install "numpy==2.0.2"
%pip install "scipy==1.16.3"

# install pyscf: as pyscf[geomopt,dispersion] gives problems, they are installed separately
%pip install --prefer-binary "pyscf==2.11.0"
%pip install "geometric==1.1"
%pip install "pyscf-dispersion==1.3.0"

%pip install "py3Dmol==2.5.3"
%pip install "pubchempy==1.0.5"
%pip install "rdkit==2025.9.1"

# !pip index versions pyscf      # check available versions
# %pip install "pyberny==0.6.3"  # install specific version
# !pip show pyberny              # check installed version

In [ ]:
#@title <small> 💻 Download subroutines.py with constants and subroutines from GitHub<small> { display-mode: "form" }

# Download the latest version of subroutines.py from the GitHub repository
# -q: run quietly (no verbose output)
# -O: save the file with the specified name (subroutines.py)
!wget -q -O subroutines.py https://raw.githubusercontent.com/daferro/Notebooks-Chemical_Equilibrium/main/subroutines.py

# Import the importlib module, which allows reloading already imported modules
import importlib

# Import the subroutines module (the downloaded Python file)
import subroutines
import subroutines as sr

# Reload the module to ensure that the most recent version is used
# This is important if the file was updated and the cell is executed again
importlib.reload(subroutines)

# Import all functions, constants, and variables from the module into the notebook namespace
from subroutines import *

# --------------------------------------------- #
def on_download_clicked(_,_case_):
    if _case_.startswith("kin"):
       init_conditions = f"{T_slider.value:.0f}K_{P_slider.value:.2f}bar_{V_slider.value:.2f}L_{yA_slider.value:.2f}"
    else:

    # update fig using last_fig
    # (a) from subroutine.py
       fig = sr.last_fig
    # (b) from this notebook
    for k in "kin,interc".split(","):
        if _case_.startswith(k): fig = last_fig

    # Can figure be downloaded?
    if fig is None:
       print("No figure yet; move a slider to generate the plot...")
       return

    # --- get file name ---
    if   _case_ == "PT"       : fname = f"equilibrium_T{T_slider.value:.0f}K_p{P_slider.value:.2f}bar.svg"
    elif _case_ == "VT"       : fname = f"equilibrium_T{T_slider.value:.0f}K_V{V_slider.value:.2f}L.svg"
    elif _case_ == "intercept": fname = f"intercept_T{T_slider.value:.0f}K_p{P_slider.value:.2f}bar.svg"
    elif _case_ == "dof"      : fname = "vib_dof.svg"
    elif _case_ == "kinPT"    : fname = f"chemkinPT_{init_conditions:s}.svg"
    elif _case_ == "kinVT"    : fname = f"chemkinVT_{init_conditions:s}.svg"
    else                      : raise Exception

    # if _case_.startswith("kin"): original_size = fig.get_size_inches()
    # if _case_.startswith("kin"): fig.set_size_inches(12,6)
    fig.savefig(fname, bbox_inches='tight',dpi=300)
    # if _case_.startswith("kin"): fig.set_size_inches(original_size)
    files.download(fname)
# --------------------------------------------- #

### **Loading the reaction**  


Execute the following cell in order to define your reaction.

In [ ]:
#@title <small><small> { display-mode: "form" }
NUS, MOLECULES = ask_for_reaction(max_nerr=3)

### **1. A Chemical Thermodynamics Perspective**  


####
We now need the thermodynamic parameters for this reaction (you may consult, for example, Atkins' *Physical Chemistry*, **2014**, Oxford University Press):

Execute the following cell and provide the required data when prompted.

In [ ]:
#@title <small><small> { display-mode: "form" }

magnitudes = []
magnitudes += [("H","formation entalphy [kJ/mol]")]
magnitudes += [("S","entropy [J/(mol K)]")]
magnitudes += [("C","molar heat capacities at constant P [J/(mol K)]")]

print(rf"We need the thermodynamical data for all the species.")

# ---- Get thermodynamical data ----
TREF = ask_for_float("First, set the reference temperature (K): ")
print("")
#------------------------------------------------------
TMIN  = max(TREF-150,0)
TMAX  = TREF+150
#------------------------------------------------------

USERDATA = {}
for molecule in MOLECULES:
    print(rf"Introduce the thermodynamical data for species: {molecule:s}")
    print("")
    USERDATA[molecule] = {}
    for key,magnitude in magnitudes:
        count = 0
        USERDATA[molecule][key] = ask_for_float(rf"    * {magnitude:47s}: ")
    print("")

# ---- Get values for the reaction ----
DHo_ref  = sum([nu_i*USERDATA[molec_i]["H"] for nu_i, molec_i in zip(NUS,MOLECULES)])
DSo_ref  = sum([nu_i*USERDATA[molec_i]["S"] for nu_i, molec_i in zip(NUS,MOLECULES)])
DCPo_ref = sum([nu_i*USERDATA[molec_i]["C"] for nu_i, molec_i in zip(NUS,MOLECULES)])

# ---- Print values for the reaction ----
print(rf"Magnitudes of interest at {TREF:.2f} K:")
print(rf"    Delta_r{{H}}^o  = {DHo_ref:+8.2f} kJ/mol")
print(rf"    Delta_r{{S}}^o  = {DSo_ref:+8.2f}  J/(mol K)")
print(rf"    Delta_r{{Cp}}^o = {DCPo_ref:+8.2f}  J/(mol K)")

# ---- Convert Entalphy of reaction to SI ----
DHo_ref *= 1000

# ---- Save data ----
REFDATA = (DHo_ref,DSo_ref,DCPo_ref,TREF)
#------------------------------------------------------

####
(a) EFFECT OF THE TEMPERATURE ON $\Delta_r G^\circ$

#####
For an ideal gas-phase reaction, the temperature dependence of the standard Gibbs free energy change, $\Delta_{r} G^\circ$, can be expressed as:

$$
\Delta_{r} G^{\circ}(T)= \Delta_{r} H^{\circ}(T_{\rm ref})-T \cdot \Delta_{r} S^{\circ}(T_{\rm ref})+\Delta_{r} C_p^\circ \cdot \left[ T-T_{\rm ref}+T \cdot \ln \left(\frac{T_{\rm ref}}{T}\right)\right]
 \tag{1}
$$

with $T_{\rm ref}$ being the reference $T$ and under the assumption that $\Delta_{r} C_p^\circ$ is independent of $T$. Notice that the corresponding equilibrium constant is then obtained as:

$$
K_p^\circ(T) = e^{-\Delta_{r} G^{\circ}(T)/(R \cdot T)}
 \tag{2}
$$

Examine the temperature dependence of $\Delta G^\circ$ (and of $K_p$) by executing the next cell. In the generated plots, the red dot indicates the value at $T_{\rm ref}$.

In [ ]:
#@title <small><small> { display-mode: "form" }
#------------------------------------------------------
T     = np.linspace(TMIN,TMAX,num=NPOINTST)
DGo_T = plot_DGo_T(T,TREF,REFDATA)
#------------------------------------------------------

#####
According to the Gibbs–Helmholtz equation, the following equality should hold:

$$
\frac{\Delta_{r} H^{\circ}}{T^2} = \frac{d}{dT}\left( -\frac{\Delta_{r} G^{\circ}}{T} \right)
\tag{3}
$$

This relationship can be tested by computing the numerical derivative of the term on the right-hand side and comparing it with the analytical expression on the left-hand side.

- The analytical expression for the enthalpy term as a function of temperature is:
$$
\Delta_{r} H^{\circ} = \Delta_{r} H^{\circ}(T_{\rm ref}) + \Delta_{r} C_{p}^\circ \cdot (T- T_{\rm ref})
\tag{4}
$$
Using this expression, we can easily plot $\Delta_{r} H^{\circ}/ T^2$.

- On the other side, the numerical derivative of $-\Delta_{r} G^{\circ}/T$ with respect to temperature can be directly computed from the data obtained in the execution of the previous cell.

Execute the next cell to check the Gibbs–Helmholtz relationship.

In [ ]:
#@title <small><small> { display-mode: "form" }
plot_gibbshelmholtz(T,DGo_T,REFDATA)

####
(b) REACTION CONDUCTED AT CONSTANT _P_ AND _T_

#####
$G$ changes dynamically as the reaction progresses. By knowing the initial conditions for the reaction, it is possible to track how $G$ varies with the extent of reaction ($\xi$), which is of special interest when the reaction takes place at constant $T$ and $p$.

Firstly, set the initial number of moles of each species involved in your reaction by executing the next cell.

In [ ]:
#@title <small><small> { display-mode: "form" }

print("Let us set the initial conditions")
print("")

print("   - indicate, first, the number of moles of each species!")
#-----------------------------------
n_0 = []
for molecule in MOLECULES:
    nmol = ask_for_float(rf"     n_mol({molecule:s}): ")
    n_0.append(nmol)
n_0 = np.array(n_0)
#-----------------------------------
xi_min,xi_max = limits_xi(n_0,NUS)
xis           = np.linspace(xi_min,xi_max,num=NPOINTSXI)
#-----------------------------------
print("")
print(rf"   min value for xi: {xi_min:+7.3f} mol")
print(rf"   max value for xi: {xi_max:+7.3f} mol")
print("")

Now, execute the cell below, set the temperature and the total pressure, and analyze how $G$ changes with $\xi$:

In [ ]:
#@title <small><small> { display-mode: "form" }

#----   Save info in a tuple    ----
fixed_args    = (MOLECULES,NUS,n_0,xis,REFDATA)
#-----------------------------------

# Enable Colab’s custom widget manager, allowing interactive ipywidgets to function correctly
output.enable_custom_widget_manager()

# -------- Sliders --------
args     = dict(layout=w.Layout(width='600px'),style={'description_width': '150px'},continuous_update=True,readout_format='.2f')
T_slider = w.FloatSlider(value=TREF,min=TMIN,max=TMAX,step=1.00,description=r'T [K]'  , **args)
P_slider = w.FloatSlider(value=1.00,min=0.01,max=3.00,step=0.01,description=r'p [bar]', **args)
ui       = w.VBox([T_slider,P_slider])

# -------- download button --------
btn      = w.Button(description='Download current figure', icon='download', button_style='primary',layout=w.Layout(width='200px', height='30px'))
btn.on_click(lambda b: on_download_clicked(b,"PT"))

# -------- slider ---> function --------
# notice that P is converted from bar to Pa with *1E5
out = w.interactive_output(lambda T, P, fixed_args: plot_DG_PT(T, P*1E5, fixed_args), {'T': T_slider, 'P': P_slider, 'fixed_args': w.fixed(fixed_args)})
display(w.VBox([ui, btn]), out)


####
(c) REACTION CONDUCTED AT CONSTANT _V_ AND _T_

#####
Another case of interest is a reaction carried out in a reactor at constant $T$ and $V$. Under these conditions, the relevant thermodynamic potential is the Helmholtz free energy (A):

$$
A = G - pV = G - nRT
\tag{7}
$$

Let us assume that the reaction vessel initially contains the number of moles that you defined previously. Execute the cell below, set the temperature and the total volume, and analyze how $A$ changes with $\xi$:


In [ ]:
#@title <small><small> { display-mode: "form" }

#----   Save info in a tuple    ----
fixed_args    = (MOLECULES,NUS,n_0,xis,REFDATA)
#-----------------------------------

# Enable Colab’s custom widget manager, allowing interactive ipywidgets to function correctly
output.enable_custom_widget_manager()

# -------- Sliders --------
args     = dict(layout=w.Layout(width='600px'),style={'description_width': '150px'},continuous_update=True,readout_format='.2f')
T_slider = w.FloatSlider(value=TREF ,min=TMIN,max=TMAX  ,step=1.00,description=r'T [K]', **args)
V_slider = w.FloatSlider(value=24.78,min=2.00,max=250.00,step=0.01,description=r'V [L]', **args)
ui       = w.VBox([T_slider,V_slider])

# -------- download button --------
btn      = w.Button(description='Download current figure', icon='download', button_style='primary',layout=w.Layout(width='200px', height='30px'))
btn.on_click(lambda b: on_download_clicked(b,"VT"))

# -------- slider ---> function --------
# V is converted from L to m3 with *1E-3
out = w.interactive_output(lambda T, V, fixed_args: plot_DA_VT(T, V*1E-3, fixed_args), {'T': T_slider, 'V': V_slider, 'fixed_args': w.fixed(fixed_args)})
display(w.VBox([ui, btn]), out)

### **2. A Statistical-Mechanical Perspective**  


#####
In the previous section, we analyzed the equilibrium using tabulated values of $\Delta_{r} H^{\circ}(T_{\rm ref})$, $\Delta_{r} S^{\circ}(T_{\rm ref})$ and $\Delta_{r} C_p^\circ(T_{\rm ref})$, which were then inserted into a macroscopic expression for $\Delta_{r} G^{\circ}(T)$. In this section, we will obtain these thermodynamic quantities from a molecular-level description using statistical mechanics.

For an ideal-gas mixture, the standard thermodynamic functions of each species can be derived from its molecular partition function, $q^\circ(T)$, which factorizes into translational, rotational, vibrational, and electronic contributions. From $q^\circ(T)$ we can compute $G^{\circ}_{m}(T)$, $H^{\circ}_{m}(T)$, $S^{\circ}_{m}(T)$, and $C^{\circ}_{p,m}(T)$ for each molecule. The reaction quantities then follow by applying the stoichiometric sum over reactants and products. In this way, the temperature dependence of
$\Delta_{r} G^{\circ}(T)$ is linked directly to molecular structure and vibrational spectra.

To start, we need initial molecular geometries for reactant(s) and product(s). These structures will be used as input for subsequent electronic-structure calculations (geometry optimization and harmonic vibrational analysis), from which we will extract vibrational frequencies and other molecular parameters required to build the partition functions.

#### _(a) Guess geometries of reactants and products_

This notebook can either retrieve the geometry of a species from PubChem[[🌐]](https://pubchem.ncbi.nlm.nih.gov/) by using its PubChem CID identifier[[🌐]](https://pubchem.ncbi.nlm.nih.gov/compound/25352); or it can build the molecule from its SMILES representation[[🌐]](https://en.wikipedia.org/wiki/Simplified_Molecular_Input_Line_Entry_System) using with the **Rdkit** toolkit[[🌐]](https://www.rdkit.org/).

Run the following cell and enter the PubChem CID or the SMILES code for each species.
Once the geometry is retrieved/generated, its **3D geometry** will be displayed in an interactive viewer that allows you to rotate and inspect the molecule freely.


In [ ]:
#@title <small><small> { display-mode: "form" }

#------------------------------------------------------
MOLECULES_ID  = {k:None for k in MOLECULES}
#------------------------------------------------------

print("Getting structures. Insert one of the following:")
print("   - its SMILES code        [e.g. CCO]")
print("   - its PubChem CID number [e.g. 702]")
print("")

# Ask for CID or SMILES
for molecule in MOLECULES:
    MOLECULES_ID[molecule] = input(rf"   * {molecule:s}: ").strip()
print("")

# Get structures
for molecule in MOLECULES:
    print(rf"   * Obtaining guess geometry for {molecule:s}")
    mol_id    = MOLECULES_ID[molecule]
    xyz_guess = files_of_interest(molecule)[0]
    for getguess in [pubchem_cid,rdkit_smiles2geom]:
        symbols,coords,smiles = getguess(mol_id)
        if symbols is not None: break
    if symbols is None: continue
    # write file
    data_2_xyz(symbols,coords,xyz_guess,smiles)
    # visualize
    view = create_visualization_xyz(xyz_guess)
    view.show()
    print("")


#####
The molecular structures obtained so far are **not** optimized; they do not correspond to the **minimum-energy geometries**.

Before performing any optimization, however, we must specify for each molecule both its total charge and the number of unpaired electrons.

Run the next cell and enter the value for these variables:

In [ ]:
#@title <small><small> { display-mode: "form" }

CHARGES,UNPAIREDS = {},{}
print("Insert the total charge (a.u.) and number of unpaired electrons for each molecule (integers only):")
print("")
for molecule in MOLECULES:
    print(rf"   * {molecule:s}")
    charge   = input("     total molecular charge [in au] = ").strip()
    unpaired = input("     number of unpaired electrons   = ").strip()
    CHARGES[molecule]   = int(charge)
    UNPAIREDS[molecule] = int(unpaired)
    print("")

#### _(b) DFT calculations_

#####

Now, we are ready to perform the quantum-chemistry calculations using **PySCF**[[🌐]](https://pyscf.org). After optimizing the geometries, we will also compute the **Hessian matrix** to verify that each structure corresponds to a true minimum (all vibrational frequencies are real), rather than a different type of stationary point. Moreover, these vibrational frequencies are required to obtain the vibrational partition function, which is essential for evaluating thermodynamic quantities.

Run the following cell to enter a DFT functional and basis set. Make sure that both are supported by PySCF. Once inserted, the quantum-chemistry calculations will be performed. Repeat the execution to try different levels.

**Note:** The DFT integration grid was set to **4** though the `DFTGRID` variable (0 - 9). Modify it in the cell if needed (big number for large mesh grids).



In [ ]:
#@title <small><small> { display-mode: "form" }

# =====================================================
# THIS VARIABLE CAN BE MODIFIED:
DFTGRID = 4
# =====================================================

#------------------------------------------------------
if "DFTDATA" not in globals(): DFTDATA = {k:{}   for k in MOLECULES}
if "KEYS"    not in globals(): KEYS    = []
#------------------------------------------------------
# Ask for functional and basis
functional = input("  insert DFT functional: ").strip()
basis      = input("  insert basis set     : ").strip()
print("")
#------------------------------------------------------
# Print information
key = (functional,basis)
print(rf" - Functional: {key[0].upper():s}")
print(rf" - Basis set : {key[1].upper():s}")
print(rf" - Grid level: {DFTGRID:d}")
print("")
#------------------------------------------------------
# Carry on calculations
t1 = time.time()
for molecule in MOLECULES:
    print(rf" * Molecule: {molecule:s}")
    DFTDATA[molecule][key] = optimize_and_freqs(molecule,UNPAIREDS[molecule],CHARGES[molecule],key[0],key[1],DFTGRID)
    print("")
    pyscf_printdata(DFTDATA[molecule][key])
    pyscf_download(molecule,key[0],key[1],DFTGRID,[1])
    print("")
    # visualization
    xyz_opt = files_of_interest(molecule,key[0],key[1],DFTGRID)[1]
    view    = create_visualization_xyz(xyz_opt)
    view.show()
    print("")
t2 = time.time()
print(" - total elapsed time: %.2f minutes"%((t2-t1)/60))
#------------------------------------------------------
# save successful key
if key not in KEYS: KEYS.append(key)
#------------------------------------------------------

#####

PySCF does not always return the correct symmetry number ($\sigma$) due to numerical inaccuracies [[🌐]](https://link.springer.com/article/10.1007/s00214-007-0328-0). Therefore, before moving to (c) and compute the partition functions, we must ensure that the symmetry numbers are correct.

Run the following cell if any symmetry number needs to be corrected.

In [ ]:
#@title <small>💻 Correct symmetry numbers <small> { display-mode: "form" }
print("Insert the rotational symmetry number for each molecule:")
print("")
for molecule in MOLECULES:
    sigma = int(input(rf"   * sigma({molecule:s}) = ").strip())
    print("")
    for functional,basis in KEYS:
        key = (functional,basis)
        try:
          sigma_old = DFTDATA[molecule][key]["rotsigma"]
          print(rf"     {functional.upper():8s}: {sigma_old:d} --> {sigma:d}")
          DFTDATA[molecule][key]["rotsigma"] = sigma
        except:
          print(rf"     data not found for {functional.upper():s}/{basis.upper():s}...")
    print("")

#### _(c) Calculation of partition functions and thermodynamic functions_


#####
We can now evaluate the partition functions of all species participating in the reaction at a given temperature. From them, the thermodynamic state functions can be determined.

In [ ]:
#@title <small><small> { display-mode: "form" }

TABLE  = " ----------------------------------------------------------------------------------------------" + "\n"
TABLE += "  FUNCTIONAL/BASIS_SET | DELTA_r{U}^o | DELTA_r{H}^o | DELTA_r{S}^o | DELTA_r{G}^o |   K_p^o   " + "\n"
TABLE += " ----------------------|--------------|--------------|--------------|--------------|-----------" + "\n"

print("Partition functions and zero-point–corrected total energies (E0 + ZPE); energies in hartree.")
print("")

TREF = ask_for_float("First, set the reference temperature (K): ")
TMIN = max(TREF-150,0)
TMAX = TREF+150
print("")

#------------------------------------------------------
if "DFT_REFTHERMO" not in globals(): DFT_REFTHERMO = {k:{}   for k in MOLECULES}
#------------------------------------------------------
for key in KEYS:
    level = rf"{key[0].upper():8s}/{key[1].upper():s}"
    allok = True

    SUBTABLE  = "    --------------------------------------------------------------------------------------------"+"\n"
    SUBTABLE += "      molecule |   pfn tr    |  pfn rot   |  pfn vib   |  pfn ele   |  pfn tot   |   E0 + ZPE   "+"\n"
    SUBTABLE += "    -----------|-------------|------------|------------|------------|---------------------------"+"\n"

    DUo,DHo,DSo,DGo  = 0.,0.,0.,0.
    for nu_i,molecule in zip(NUS,MOLECULES):

        output_frq = files_of_interest(molecule,key[0],key[1],DFTGRID)[3]
        if not os.path.exists(output_frq):
           allok = False
           break
        # calculate thermodynamics
        try: Uo, Ho, So, Go, line = compute_thermodynamics(TREF,molecule,key,DFTDATA)
        except: allok = False; break
        SUBTABLE += rf"      {molecule:8s} | " + line + "\n"

        # reaction magnitudes
        DUo += nu_i * Uo
        DHo += nu_i * Ho
        DSo += nu_i * So
        DGo += nu_i * Go
    SUBTABLE += "    --------------------------------------------------------------------------------------------"+"\n"
    if not allok: continue
    print(rf"    ==> {level:s} <==")
    print("")
    print(SUBTABLE)
    # Equilibrium constant
    Kp = np.exp(-DGo/k_B/TREF)
    # Print info
    if   Kp > 1E+4: ff = "9.2E"
    elif Kp < 1E-3: ff = "9.2E"
    else          : ff = "9.3f"
    TABLE += rf"  {level:20s} | {DUo*NA/1000:12.2f} | {DHo*NA/1000:12.2f} | {DSo*NA:12.2f} | {DGo*NA/1000:12.2f} | {Kp:{ff:s}} " + "\n"
    # save reaction magnitudes
    DFT_REFTHERMO[key] = TREF,DUo*NA,DHo*NA,DSo*NA,DGo*NA
TABLE += " ----------------------------------------------------------------------------------------------" + "\n"
TABLE += "\n"
TABLE += "     units of DELTA_r{U}^o ==> kJ/mol"   + "\n"
TABLE += "     units of DELTA_r{H}^o ==> kJ/mol"   + "\n"
TABLE += "     units of DELTA_r{S}^o ==>  J/mol/K" + "\n"
TABLE += "     units of DELTA_r{G}^o ==> kJ/mol"   + "\n"

print("")
print(TABLE)
print("")

#### _(d) Temperature dependence of $\Delta_{r}G^\circ$: $\Delta_{r}C_{p}^\circ$ and the vibrational degrees of freedom_


#####

The calculation of $\Delta_{r}G^\circ$ in (c) can be performed at temperatures other than $T_{\rm ref}$. As you may recall from the first part of this Notebook, this temperature dependence should behaves:

$$
\Delta_{r} G^{\circ}(T) = \Delta_{r} H^{\circ}(T_{\rm ref})-T \cdot \Delta_{r} S^{\circ}(T_{\rm ref})+\Delta_{r} C_p^\circ \cdot \left[ T-T_{\rm ref}+T \cdot \ln \left(\frac{T_{\rm ref}}{T}\right)\right]
\tag{1}
$$

under the assumption that $\Delta_{r}C_{p}^\circ$ is constant. Note that you already calculated all quantities at the reference temperature ($T_{\rm ref}$), except for $\Delta_{r}C_{p}^\circ$ itself.

Next, we will obtain the value that best reproduces the numerical results obtained for $\Delta_{r}G^\circ$.

#####

The optimal value of $\Delta_{r} C_p^\circ$ can be obtained directly by fitting the computed $\Delta_{r} G^\circ(T)$ values (calculated using Statistical Thermodynamics) to Eq. (1). In other words, instead of choosing  $\Delta_{r} C_p^\circ$ manually, one could determine the value that minimizes the deviation between the analytical expression, Eq. (1), and the data, yielding the best agreement across the temperature range.

To do this, run the following cell.

In [ ]:
#@title <small><small> { display-mode: "form" }

#------------------------------------------------------------------------------#
T = np.linspace(TMIN,TMAX,num=NPOINTST)
#------------------------------------------------------------------------------#
for key_i in KEYS:
    # Print level of calculation
    level = rf"{key_i[0].upper():8s}/{key_i[1].upper():s}"
    print(rf"==> {level:s} <==" + "\n")
    # reference data
    TREF, DUo_qc, DHo_qc, DSo_qc, DGo_qc = DFT_REFTHERMO[key_i]

    # (a) Calculate D_{r}G^o at different temperatures using Statistical Thermodynamics
    print(rf"    Calculating Delta_r{{G}}^o between {TMIN:.2f} and {TMAX:.2f} K using Statistical Thermodynamics...")
    DGo_T = []
    for Ti in list(T):
        DGo_i = [nu_i * compute_thermodynamics(Ti,molecule,key_i,DFTDATA)[3] for nu_i,molecule in zip(NUS,MOLECULES)]
        DGo_T.append(float(sum(DGo_i)*NA))
    DGo_T = np.array(DGo_T)
    print("")

    # (b) best value of Cp of reaction
    print(rf"     - fitting to Eq. (1) using values at T_ref = {TREF:.2f} K...")
    res       = minimize(sum_squared_errors,np.array([0.0]),args=(T,DGo_T,TREF,DHo_qc,DSo_qc))
    dof_best  = res.x[0]
    DGo_model = get_DGo(T,(DHo_qc,DSo_qc,dof_best*R,TREF))
    RMS       = calculate_rms(DGo_T,DGo_model)/1000
    print(rf"     - best fit obtained for Delta_{{r}}C_{{p}}^o = {dof_best:.2f} * R   [RMS = {RMS:.3f} kJ/mol]")
    print("")

    # (c) Plotting
    plot_DGo_T_statmech(T,DGo_T,DGo_model,dof_best*R,TREF,DGo_qc,key_i)
    print()

    # (d) value of Cp of reaction through degrees of freedom
    print("    -------------------------------------------")
    print("     Molecule | rot dof  | vib dof  |  C_{p,m} ")
    print("    -------------------------------------------")
    DCP = 0.0
    for i,molecule in enumerate(MOLECULES):
        # check if data is available
        if molecule not in DFTDATA          : continue
        if key_i    not in DFTDATA[molecule]: continue
        # rotational dofs
        if DFTDATA[molecule][key_i]["islinear"]: rdof = 2
        else                                   : rdof = 3
        # calculate vdof and Cp
        freqs_cm = [ii/100 for ii in DFTDATA[molecule][key_i]["freqs"]]
        vdof     = sum([vib_contri_avera(freq_cm,TMIN,TMAX) for freq_cm in freqs_cm])
        Cp       = R + (3+rdof+2*vdof)/2*R
        DCP     += Cp*NUS[i]
        print(rf"     {molecule:7s}  |  {rdof:6d}  |  {vdof:6.3f}  | {Cp/R:6.3f}*R  ")
    print("    -------------------------------------------")
    print(rf"                 ==> Delta_r{{C_p}}^o = {DCP/R:6.2f}*R")
    print("")
